# 📋 FILTER DATA REGISTRASI PESERTA SERTIFIKASI
## MCF (Azure AI-900) & MOS (Office 2019)

---
**Deskripsi Program:**
- Memfilter data peserta dari file `peserta mcf.csv` dan `peserta mos.csv`
- Cross-check dengan database Certiport (`certiport.csv`)
- Memberikan rekomendasi APPROVE / JANGAN APPROVE berdasarkan status ujian sebelumnya

**Input Files:**
1. `peserta mcf.csv` - Data pendaftar program MCF (Azure AI-900)
2. `peserta mos.csv` - Data pendaftar program MOS (Office 2019)
3. `certiport.csv` - Database hasil ujian dari Certiport

**Output Files:**
- `APPROVE_INI_belum_pernah_ujian.csv`
- `JANGAN_APPROVE_sudah_ujian.csv`
- `HASIL_FILTER_PESERTA_LENGKAP.xlsx`

---

## 1️⃣ Import Library & Setup

In [30]:
# ============================================================
# IMPORT LIBRARY YANG DIPERLUKAN
# ============================================================
import pandas as pd
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import warnings
warnings.filterwarnings('ignore')

print("✅ Library berhasil diimport!")
print("   - pandas: untuk manipulasi data")
print("   - fuzzywuzzy: untuk fuzzy string matching")

✅ Library berhasil diimport!
   - pandas: untuk manipulasi data
   - fuzzywuzzy: untuk fuzzy string matching


## 2️⃣ Load & Preview Data

In [31]:
# ============================================================
# LOAD DATA DARI FILE CSV (dari folder parent)
# ============================================================

# Load data peserta MCF dan MOS (dari folder parent)
peserta_mcf_df = pd.read_csv('../peserta mcf.csv')
peserta_mos_df = pd.read_csv('../peserta mos.csv')

# Load data Certiport (dari folder parent, skip 4 baris header)
certiport_df = pd.read_csv('../certiport.csv', skiprows=4)
certiport_df = certiport_df.dropna(axis=1, how='all')  # Hapus kolom kosong

# Tambahkan kolom sumber untuk identifikasi
peserta_mcf_df['Sumber'] = 'MCF'
peserta_mos_df['Sumber'] = 'MOS'

# Gabungkan kedua data peserta
peserta_df = pd.concat([peserta_mcf_df, peserta_mos_df], ignore_index=True)

# Filter data certiport berdasarkan jenis ujian
certiport_mcf = certiport_df[certiport_df['Exam'].str.contains('AI-900', na=False)].copy()
certiport_mos = certiport_df[certiport_df['Exam'].str.contains('Office 2019', na=False)].copy()

# ============================================================
# TAMPILKAN RINGKASAN DATA
# ============================================================
print(f"{'='*70}")
print("📊 RINGKASAN DATA YANG DILOAD")
print(f"{'='*70}")
print(f"\n📁 DATA PESERTA PENDAFTAR:")
print(f"   • Peserta MCF (Azure AI-900) : {len(peserta_mcf_df):>4} orang")
print(f"   • Peserta MOS (Office 2019)  : {len(peserta_mos_df):>4} orang")
print(f"   • TOTAL PESERTA              : {len(peserta_df):>4} orang")

print(f"\n📁 DATA CERTIPORT (HASIL UJIAN):")
print(f"   • Total Record               : {len(certiport_df):>4}")
print(f"   • Data MCF (AI-900)          : {len(certiport_mcf):>4}")
print(f"   • Data MOS (Office 2019)     : {len(certiport_mos):>4}")

print(f"\n📊 Breakdown Status Peserta:")
print(peserta_df['Status'].value_counts().to_string())

📊 RINGKASAN DATA YANG DILOAD

📁 DATA PESERTA PENDAFTAR:
   • Peserta MCF (Azure AI-900) :   91 orang
   • Peserta MOS (Office 2019)  :  146 orang
   • TOTAL PESERTA              :  237 orang

📁 DATA CERTIPORT (HASIL UJIAN):
   • Total Record               : 4853
   • Data MCF (AI-900)          :  897
   • Data MOS (Office 2019)     : 3445

📊 Breakdown Status Peserta:
Status
APPROVED        165
NOT APPROVED     72


## 3️⃣ Fungsi Fuzzy Matching

In [32]:
# ============================================================
# FUNGSI-FUNGSI UNTUK FUZZY MATCHING (IMPROVED)
# ============================================================

def normalize_name(name):
    """Normalisasi nama: uppercase, hapus karakter khusus kecuali - dan ., rapikan spasi"""
    if pd.isna(name):
        return ""
    # Keep alphanumeric, spaces, dash, and dot
    cleaned = ''.join(c for c in str(name) if c.isalnum() or c.isspace() or c in '.-')
    return ' '.join(cleaned.upper().strip().split())

def reverse_name(name):
    """Balik urutan nama (untuk antisipasi nama terbalik)"""
    parts = name.split()
    if len(parts) >= 2:
        return ' '.join(parts[::-1])
    return name

def count_matching_words(name1, name2, min_word_length=2):
    """Hitung jumlah kata yang sama antara dua nama"""
    words1 = set(w for w in name1.split() if len(w) >= min_word_length)
    words2 = set(w for w in name2.split() if len(w) >= min_word_length)
    return len(words1.intersection(words2))

def get_min_matching_words(name):
    """
    Tentukan minimal kata yang harus cocok berdasarkan jumlah kata dalam nama:
    - 4+ kata: minimal 3 kata cocok
    - 3 kata: minimal 2 kata cocok
    - 2 kata: minimal 2 kata cocok
    - 1 kata: harus ada tanda - atau . di nama (special case)
    """
    words = [w for w in name.split() if len(w) >= 2]
    word_count = len(words)
    
    if word_count >= 4:
        return 3
    elif word_count == 3:
        return 2
    elif word_count == 2:
        return 2
    else:
        return 1  # Special handling for single word names

def is_single_word_match(name1, name2):
    """
    Untuk nama 1 kata, cek apakah ada tanda - atau . yang menunjukkan kecocokan
    Contoh: 'ABDUL-RAHMAN' atau 'A.RAHMAN'
    """
    # Cek apakah nama mengandung - atau .
    has_special_char = '-' in name1 or '.' in name1 or '-' in name2 or '.' in name2
    if not has_special_char:
        return False
    
    # Normalize tanpa - dan . untuk perbandingan
    clean1 = name1.replace('-', ' ').replace('.', ' ').upper()
    clean2 = name2.replace('-', ' ').replace('.', ' ').upper()
    
    # Cek kecocokan
    score = fuzz.token_set_ratio(clean1, clean2)
    return score >= 85

def find_best_match(peserta_name, certiport_names, threshold=80):
    """
    Mencari kecocokan nama terbaik dengan aturan:
    - 4+ kata: minimal 3 kata cocok
    - 3 kata: minimal 2 kata cocok
    - 2 kata: minimal 2 kata cocok
    - 1 kata: harus ada tanda - atau . dan match
    """
    peserta_normalized = normalize_name(peserta_name)
    min_words_required = get_min_matching_words(peserta_normalized)
    peserta_word_count = len([w for w in peserta_normalized.split() if len(w) >= 2])
    
    # Cek exact match
    if peserta_normalized in certiport_names:
        return peserta_normalized, 100, "Exact Match"
    
    # Cek nama terbalik
    reversed_name_str = reverse_name(peserta_normalized)
    if reversed_name_str in certiport_names:
        return reversed_name_str, 100, "Exact Match (Nama Terbalik)"
    
    best_score = 0
    best_match = None
    match_type = ""
    best_word_count = 0
    
    for cert_name in certiport_names:
        # Special handling untuk nama 1 kata
        if peserta_word_count == 1:
            if is_single_word_match(peserta_normalized, cert_name):
                return cert_name, 90, "Single Word Match (dengan tanda - atau .)"
            continue
        
        matching_words = count_matching_words(peserta_normalized, cert_name)
        
        # Cek minimum kata yang harus cocok
        if matching_words < min_words_required:
            continue
        
        # Berbagai metode fuzzy matching
        score1 = fuzz.ratio(peserta_normalized, cert_name)
        score2 = fuzz.token_set_ratio(peserta_normalized, cert_name)
        score3 = fuzz.token_sort_ratio(peserta_normalized, cert_name)
        score4 = fuzz.partial_ratio(peserta_normalized, cert_name)
        
        max_score = max(score1, score2, score3, score4)
        
        if matching_words > best_word_count or (matching_words == best_word_count and max_score > best_score):
            best_score = max_score
            best_match = cert_name
            best_word_count = matching_words
            if max_score == score1:
                match_type = "Fuzzy Ratio"
            elif max_score == score2:
                match_type = "Token Set Ratio"
            elif max_score == score3:
                match_type = "Token Sort Ratio"
            else:
                match_type = "Partial Ratio"
    
    if best_score >= threshold and best_word_count >= min_words_required:
        return best_match, best_score, f"{match_type} ({best_word_count}/{min_words_required} kata cocok)"
    
    return None, best_score, "Tidak Ditemukan"

print("✅ Fungsi matching (IMPROVED) berhasil dibuat!")
print("\n📋 Aturan Minimal Kata Cocok:")
print("   • Nama 4+ kata → minimal 3 kata cocok")
print("   • Nama 3 kata  → minimal 2 kata cocok")
print("   • Nama 2 kata  → minimal 2 kata cocok")
print("   • Nama 1 kata  → harus ada tanda - atau . dan match")

✅ Fungsi matching (IMPROVED) berhasil dibuat!

📋 Aturan Minimal Kata Cocok:
   • Nama 4+ kata → minimal 3 kata cocok
   • Nama 3 kata  → minimal 2 kata cocok
   • Nama 2 kata  → minimal 2 kata cocok
   • Nama 1 kata  → harus ada tanda - atau . dan match


## 4️⃣ Proses Cross-Check (Hanya NOT APPROVED)

In [33]:
# ============================================================
# PISAHKAN DATA BERDASARKAN STATUS
# ============================================================
print(f"{'='*70}")
print("📋 PEMISAHAN DATA BERDASARKAN STATUS REGISTRASI")
print(f"{'='*70}")

approved_df = peserta_df[peserta_df['Status'] == 'APPROVED'].copy()
not_approved_df = peserta_df[peserta_df['Status'] == 'NOT APPROVED'].copy()

print(f"\n   ✅ Status APPROVED     : {len(approved_df):>4} orang")
print(f"   ⏳ Status NOT APPROVED : {len(not_approved_df):>4} orang")

# ============================================================
# PROSES CROSS-CHECK UNTUK NOT APPROVED
# ============================================================
print(f"\n{'='*70}")
print("🔍 PROSES CROSS-CHECK UNTUK PESERTA NOT APPROVED")
print(f"{'='*70}")

# Buat Full Name dan normalisasi untuk Certiport
certiport_mcf['Full Name'] = certiport_mcf['First Name'].fillna('') + ' ' + certiport_mcf['Last Name'].fillna('')
certiport_mos['Full Name'] = certiport_mos['First Name'].fillna('') + ' ' + certiport_mos['Last Name'].fillna('')

certiport_mcf_names = [normalize_name(name) for name in certiport_mcf['Full Name'].tolist()]
certiport_mos_names = [normalize_name(name) for name in certiport_mos['Full Name'].tolist()]

# Proses matching untuk NOT APPROVED
results = []
for idx, row in not_approved_df.iterrows():
    peserta_name = row['Nama']
    sumber_data = row['Sumber']
    
    # Pilih database certiport sesuai sumber
    certiport_names = certiport_mcf_names if sumber_data == 'MCF' else certiport_mos_names
    
    match, score, match_type = find_best_match(peserta_name, certiport_names)
    
    results.append({
        'Nama Peserta': peserta_name,
        'NIM': row['NIM'],
        'Jurusan': row['Jurusan'],
        'Sumber Data': sumber_data,
        'Program': row['Program Dipilih'],
        'Status Registrasi': row['Status'],
        'Nama di Certiport': match if match else '-',
        'Skor Kecocokan': score,
        'Tipe Match': match_type,
        'Status Certiport': '✅ DITEMUKAN' if match else '❌ TIDAK DITEMUKAN',
        'Rekomendasi': '❌ JANGAN APPROVE - Sudah Pernah Ujian' if match else '✅ APPROVE - Belum Pernah Ujian'
    })

results_df = pd.DataFrame(results)

print(f"\n✅ Proses cross-check selesai untuk {len(results)} peserta NOT APPROVED!")
print(f"\n📊 Hasil Cross-Check NOT APPROVED:")
print(results_df['Status Certiport'].value_counts().to_string())

# ============================================================
# PROSES VERIFIKASI UNTUK APPROVED (FILTER TAMBAHAN)
# ============================================================
print(f"\n{'='*70}")
print("🔍 VERIFIKASI PESERTA APPROVED - CEK APAKAH BENAR TIDAK ADA DI CERTIPORT")
print(f"{'='*70}")

approved_results = []
for idx, row in approved_df.iterrows():
    peserta_name = row['Nama']
    sumber_data = row['Sumber']
    
    # Pilih database certiport sesuai sumber
    certiport_names = certiport_mcf_names if sumber_data == 'MCF' else certiport_mos_names
    
    match, score, match_type = find_best_match(peserta_name, certiport_names)
    
    if match:
        status = '⚠️ PERLU DICEK - ADA DI CERTIPORT!'
        warning = True
    else:
        status = '✅ OK - Tidak ada di Certiport'
        warning = False
    
    approved_results.append({
        'Nama Peserta': peserta_name,
        'NIM': row['NIM'],
        'Jurusan': row['Jurusan'],
        'Sumber Data': sumber_data,
        'Program': row['Program Dipilih'],
        'Status Registrasi': row['Status'],
        'Nama di Certiport': match if match else '-',
        'Skor Kecocokan': score,
        'Tipe Match': match_type,
        'Status Verifikasi': status,
        'Perlu Dicek': warning
    })

approved_filter_df = pd.DataFrame(approved_results)
approved_warning_df = approved_filter_df[approved_filter_df['Perlu Dicek'] == True].copy()
approved_ok_df = approved_filter_df[approved_filter_df['Perlu Dicek'] == False].copy()

print(f"\n✅ Verifikasi APPROVED selesai!")
print(f"\n📊 Hasil Verifikasi APPROVED:")
print(f"   ✅ OK (Tidak ada di Certiport)      : {len(approved_ok_df):>4} peserta")
print(f"   ⚠️  PERLU DICEK (Ada di Certiport)  : {len(approved_warning_df):>4} peserta")

if len(approved_warning_df) > 0:
    print(f"\n⚠️ PERHATIAN! {len(approved_warning_df)} peserta APPROVED ternyata SUDAH ADA di Certiport!")
    print(f"   Peserta ini mungkin TIDAK SEHARUSNYA di-approve.")

📋 PEMISAHAN DATA BERDASARKAN STATUS REGISTRASI

   ✅ Status APPROVED     :  165 orang
   ⏳ Status NOT APPROVED :   72 orang

🔍 PROSES CROSS-CHECK UNTUK PESERTA NOT APPROVED

✅ Proses cross-check selesai untuk 72 peserta NOT APPROVED!

📊 Hasil Cross-Check NOT APPROVED:
Status Certiport
✅ DITEMUKAN          62
❌ TIDAK DITEMUKAN    10

🔍 VERIFIKASI PESERTA APPROVED - CEK APAKAH BENAR TIDAK ADA DI CERTIPORT

✅ Verifikasi APPROVED selesai!

📊 Hasil Verifikasi APPROVED:
   ✅ OK (Tidak ada di Certiport)      :  164 peserta
   ⚠️  PERLU DICEK (Ada di Certiport)  :    1 peserta

⚠️ PERHATIAN! 1 peserta APPROVED ternyata SUDAH ADA di Certiport!
   Peserta ini mungkin TIDAK SEHARUSNYA di-approve.


## 5️⃣ Hasil & Rekomendasi

In [34]:
# ============================================================
# PISAHKAN HASIL BERDASARKAN REKOMENDASI
# ============================================================

# Peserta yang ditemukan di Certiport (sudah pernah ujian)
found_df = results_df[results_df['Status Certiport'] == '✅ DITEMUKAN'].copy()

# Peserta yang tidak ditemukan di Certiport (belum pernah ujian)
not_found_df = results_df[results_df['Status Certiport'] == '❌ TIDAK DITEMUKAN'].copy()

# ============================================================
# TAMPILKAN HASIL
# ============================================================
print(f"{'='*80}")
print("✅ PESERTA YANG DIREKOMENDASIKAN UNTUK APPROVE (Belum Pernah Ujian)")
print(f"{'='*80}")
print(f"Total: {len(not_found_df)} peserta\n")
if len(not_found_df) > 0:
    print(not_found_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Program', 'Rekomendasi']].to_string(index=False))

print(f"\n{'='*80}")
print("❌ PESERTA YANG JANGAN DI-APPROVE (Sudah Pernah Ujian)")
print(f"{'='*80}")
print(f"Total: {len(found_df)} peserta\n")
if len(found_df) > 0:
    print(found_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Nama di Certiport', 'Skor Kecocokan', 'Rekomendasi']].to_string(index=False))

✅ PESERTA YANG DIREKOMENDASIKAN UNTUK APPROVE (Belum Pernah Ujian)
Total: 10 peserta

                     Nama Peserta       NIM Sumber Data           Program                    Rekomendasi
     MUHAMMAD FADHIL BIMA JULIANO 202231025         MCF MCF: Azure AI-900 ✅ APPROVE - Belum Pernah Ujian
        ANDI MUHAMMAD NABIL ALKAF 202231112         MOS         MOS: Word ✅ APPROVE - Belum Pernah Ujian
        KOMANG ARYA DIVA PRAMARTA 202211008         MOS        MOS: Excel ✅ APPROVE - Belum Pernah Ujian
     I MADE DWI MAHA PUTRA SUJANA 202211012         MOS        MOS: Excel ✅ APPROVE - Belum Pernah Ujian
        RADEN MUHAMMAD ASYAM RAFI 202211042         MOS         MOS: Word ✅ APPROVE - Belum Pernah Ujian
      ROVAN LUWID ERVAN SIHOMBING 202212002         MOS        MOS: Excel ✅ APPROVE - Belum Pernah Ujian
ROSEMAIN ERI ASTUTI JATNIKA PUTRI 202242028         MOS         MOS: Word ✅ APPROVE - Belum Pernah Ujian
     MUHAMMAD WAHYU GUSTIAN HARUN 202214039         MOS         MOS: Word 

## 6️⃣ Deteksi Peserta Pernah Gagal

In [35]:
# ============================================================
# DETEKSI PESERTA YANG PERNAH GAGAL UJIAN
# ============================================================
print(f"{'='*70}")
print("⚠️  DETEKSI PESERTA YANG PERNAH GAGAL UJIAN")
print(f"{'='*70}")

all_certiport = pd.concat([certiport_mcf, certiport_mos], ignore_index=True)
daftar_lagi_df = pd.DataFrame()

if 'Result' in all_certiport.columns:
    failed_students = all_certiport[all_certiport['Result'] == 'Fail'].copy()
    
    if len(failed_students) > 0:
        failed_students['Full Name Normalized'] = failed_students.apply(
            lambda x: normalize_name(f"{x['First Name']} {x['Last Name']}"), axis=1
        )
        
        not_approved_df_temp = not_approved_df.copy()
        not_approved_df_temp['Nama Normalized'] = not_approved_df_temp['Nama'].apply(normalize_name)
        
        daftar_lagi = []
        for idx, row in not_approved_df_temp.iterrows():
            nama_normalized = row['Nama Normalized']
            sumber = row['Sumber']
            
            if sumber == 'MCF':
                failed_to_check = failed_students[failed_students['Exam'].str.contains('AI-900', na=False)]
            else:
                failed_to_check = failed_students[failed_students['Exam'].str.contains('Office 2019', na=False)]
            
            for _, failed_row in failed_to_check.iterrows():
                failed_name = failed_row['Full Name Normalized']
                score = fuzz.token_set_ratio(nama_normalized, failed_name)
                if score >= 85 or nama_normalized == failed_name:
                    daftar_lagi.append({
                        'Nama Peserta': row['Nama'],
                        'NIM': row['NIM'],
                        'Sumber Data': sumber,
                        'Status': '⚠️ PERNAH GAGAL - DAFTAR LAGI',
                        'Nama di Certiport': f"{failed_row['First Name']} {failed_row['Last Name']}",
                        'Exam Sebelumnya': failed_row['Exam'],
                        'Skor Sebelumnya': failed_row.get('Score', '-'),
                        'Keputusan': '❓ PERLU REVIEW MANUAL'
                    })
                    break
        
        if len(daftar_lagi) > 0:
            daftar_lagi_df = pd.DataFrame(daftar_lagi)
            print(f"\n⚠️  PERHATIAN: {len(daftar_lagi_df)} peserta PERNAH GAGAL dan DAFTAR LAGI!")
            print(daftar_lagi_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Skor Sebelumnya', 'Keputusan']].to_string(index=False))
        else:
            print("\n✅ Tidak ada peserta NOT APPROVED yang pernah gagal ujian sebelumnya")
    else:
        print("\n✅ Tidak ada data peserta yang gagal di Certiport")
else:
    print("\n⚠️ Kolom 'Result' tidak ditemukan di data Certiport")

⚠️  DETEKSI PESERTA YANG PERNAH GAGAL UJIAN

⚠️  PERHATIAN: 61 peserta PERNAH GAGAL dan DAFTAR LAGI!
                      Nama Peserta       NIM Sumber Data  Skor Sebelumnya             Keputusan
              RAIHAN CANDRA IRAWAN 202211117         MOS            668.0 ❓ PERLU REVIEW MANUAL
                   MUHAMMAD RAIHAN 202241017         MOS            636.0 ❓ PERLU REVIEW MANUAL
MUHAMMAD REVIANSYAH DANENDRA PUTRA 202131160         MOS            152.0 ❓ PERLU REVIEW MANUAL
   DHEA HERAWATI INDAH PUTRI ERWIN 202231088         MOS            127.0 ❓ PERLU REVIEW MANUAL
               BINTAR ANDIKA PUTRA 202211113         MOS            547.0 ❓ PERLU REVIEW MANUAL
          RACHMAD FAQIH AFGIANSYAH 202112050         MOS            426.0 ❓ PERLU REVIEW MANUAL
                     JIHAN DIOVANT 202212037         MOS            578.0 ❓ PERLU REVIEW MANUAL
               NADELLA EZI SABRINA 202232010         MOS            147.0 ❓ PERLU REVIEW MANUAL
            JOHN ANDREW TAMPUBOLON 

## 7️⃣ Dashboard Summary

In [36]:
# ============================================================
# DASHBOARD SUMMARY
# ============================================================
print(f"\n{'='*90}")
print(" " * 30 + "📊 DASHBOARD SUMMARY")
print(f"{'='*90}")

total_peserta = len(peserta_df)
total_not_approved = len(not_approved_df)

print(f"""
┌────────────────────────────────────────────────────────────────────────────────┐
│                           DATA PENDAFTAR                                       │
├────────────────────────────────────────────────────────────────────────────────┤
│  Total Pendaftar            : {total_peserta:>5} peserta                                   │
│    ├─ MCF (Azure AI-900)    : {len(peserta_mcf_df):>5} peserta ({len(peserta_mcf_df)/total_peserta*100:>5.1f}%)                    │
│    └─ MOS (Office 2019)     : {len(peserta_mos_df):>5} peserta ({len(peserta_mos_df)/total_peserta*100:>5.1f}%)                    │
├────────────────────────────────────────────────────────────────────────────────┤
│                         STATUS REGISTRASI                                      │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Status APPROVED         : {len(approved_df):>5} peserta ({len(approved_df)/total_peserta*100:>5.1f}%)                    │
│  ⏳ Status NOT APPROVED     : {len(not_approved_df):>5} peserta ({len(not_approved_df)/total_peserta*100:>5.1f}%)                    │
├────────────────────────────────────────────────────────────────────────────────┤
│                  HASIL CROSS-CHECK (NOT APPROVED)                              │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Ditemukan (Sudah Ujian) : {len(found_df):>5} peserta  → JANGAN APPROVE              │
│  ❌ Tidak Ditemukan         : {len(not_found_df):>5} peserta  → APPROVE INI                │
│  ⚠️  Pernah Gagal           : {len(daftar_lagi_df):>5} peserta  → PERLU REVIEW              │
├────────────────────────────────────────────────────────────────────────────────┤
│                  VERIFIKASI APPROVED (APPROVED FILTER)                         │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ OK (Tidak di Certiport) : {len(approved_ok_df):>5} peserta  → BENAR APPROVED           │
│  ⚠️  PERLU DICEK (Ada)      : {len(approved_warning_df):>5} peserta  → SEHARUSNYA TDK APPROVE   │
└────────────────────────────────────────────────────────────────────────────────┘
""")

print(f"\n🎯 REKOMENDASI AKHIR:")
print(f"   ✅ APPROVE     : {len(not_found_df)} peserta (belum pernah ujian)")
print(f"   ❌ JANGAN      : {len(found_df)} peserta (sudah pernah ujian)")
if len(daftar_lagi_df) > 0:
    print(f"   ⚠️ REVIEW     : {len(daftar_lagi_df)} peserta (pernah gagal)")
if len(approved_warning_df) > 0:
    print(f"   🔴 PERHATIAN  : {len(approved_warning_df)} peserta APPROVED tapi ADA di Certiport!")


                              📊 DASHBOARD SUMMARY

┌────────────────────────────────────────────────────────────────────────────────┐
│                           DATA PENDAFTAR                                       │
├────────────────────────────────────────────────────────────────────────────────┤
│  Total Pendaftar            :   237 peserta                                   │
│    ├─ MCF (Azure AI-900)    :    91 peserta ( 38.4%)                    │
│    └─ MOS (Office 2019)     :   146 peserta ( 61.6%)                    │
├────────────────────────────────────────────────────────────────────────────────┤
│                         STATUS REGISTRASI                                      │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Status APPROVED         :   165 peserta ( 69.6%)                    │
│  ⏳ Status NOT APPROVED     :    72 peserta ( 30.4%)                    │
├─────────────────────────────────────────────────────────────────

## 8️⃣ Export Hasil ke CSV

In [37]:
# ============================================================
# EXPORT HASIL KE FILE CSV
# ============================================================
print(f"{'='*70}")
print("💾 EXPORT HASIL KE FILE CSV")
print(f"{'='*70}")

# Export peserta yang direkomendasikan APPROVE
not_found_df.to_csv('APPROVE_INI_belum_pernah_ujian.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ APPROVE_INI_belum_pernah_ujian.csv")
print(f"   → {len(not_found_df)} peserta yang LAYAK DI-APPROVE")

# Export peserta yang JANGAN di-approve
found_df.to_csv('JANGAN_APPROVE_sudah_ujian.csv', index=False, encoding='utf-8-sig')
print(f"\n❌ JANGAN_APPROVE_sudah_ujian.csv")
print(f"   → {len(found_df)} peserta yang JANGAN DI-APPROVE")

# Export peserta APPROVED (langsung diterima)
approved_df.to_csv('peserta_approved.csv', index=False, encoding='utf-8-sig')
print(f"\n📋 peserta_approved.csv")
print(f"   → {len(approved_df)} peserta dengan status APPROVED")

# Export semua hasil cross-check
results_df.to_csv('hasil_crosscheck_not_approved.csv', index=False, encoding='utf-8-sig')
print(f"\n📊 hasil_crosscheck_not_approved.csv")
print(f"   → Semua hasil cross-check ({len(results_df)} peserta)")

# Export peserta yang perlu review
if len(daftar_lagi_df) > 0:
    daftar_lagi_df.to_csv('PERHATIAN_peserta_gagal_daftar_lagi.csv', index=False, encoding='utf-8-sig')
    print(f"\n⚠️ PERHATIAN_peserta_gagal_daftar_lagi.csv")
    print(f"   → {len(daftar_lagi_df)} peserta yang PERLU REVIEW")

💾 EXPORT HASIL KE FILE CSV

✅ APPROVE_INI_belum_pernah_ujian.csv
   → 10 peserta yang LAYAK DI-APPROVE

❌ JANGAN_APPROVE_sudah_ujian.csv
   → 62 peserta yang JANGAN DI-APPROVE

📋 peserta_approved.csv
   → 165 peserta dengan status APPROVED

📊 hasil_crosscheck_not_approved.csv
   → Semua hasil cross-check (72 peserta)

⚠️ PERHATIAN_peserta_gagal_daftar_lagi.csv
   → 61 peserta yang PERLU REVIEW


## 9️⃣ Export ke Excel dengan Multiple Sheets

In [38]:
# ============================================================
# EXPORT KE EXCEL DENGAN FORMATTING
# ============================================================
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

print(f"{'='*70}")
print("📊 EXPORT KE EXCEL DENGAN MULTIPLE SHEETS")
print(f"{'='*70}")

wb = Workbook()

# Style definitions
header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)
approve_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
reject_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
warning_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

def style_worksheet(ws, highlight_col=None, approve_val=None, reject_val=None):
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center')
        cell.border = thin_border
    
    for row_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
        for cell in row:
            cell.border = thin_border
        if highlight_col:
            cell_val = str(ws.cell(row=row_idx, column=highlight_col).value)
            fill = None
            if approve_val and approve_val in cell_val:
                fill = approve_fill
            elif reject_val and reject_val in cell_val:
                fill = reject_fill
            elif '⚠️' in cell_val or 'REVIEW' in cell_val:
                fill = warning_fill
            if fill:
                for cell in row:
                    cell.fill = fill
    
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 50)

# Sheet 1: Dashboard Summary
ws1 = wb.active
ws1.title = "Dashboard Summary"
summary = [
    ["KATEGORI", "JUMLAH", "PERSENTASE", "STATUS"],
    ["Total Pendaftar", len(peserta_df), "100%", "ℹ️"],
    ["  ├─ MCF (Azure)", len(peserta_mcf_df), f"{len(peserta_mcf_df)/len(peserta_df)*100:.1f}%", ""],
    ["  └─ MOS (Office)", len(peserta_mos_df), f"{len(peserta_mos_df)/len(peserta_df)*100:.1f}%", ""],
    ["", "", "", ""],
    ["Status APPROVED", len(approved_df), f"{len(approved_df)/len(peserta_df)*100:.1f}%", "✅"],
    ["Status NOT APPROVED", len(not_approved_df), f"{len(not_approved_df)/len(peserta_df)*100:.1f}%", "⏳"],
    ["", "", "", ""],
    ["Sudah Ujian (JANGAN APPROVE)", len(found_df), f"{len(found_df)/len(not_approved_df)*100:.1f}%" if len(not_approved_df) > 0 else "0%", "❌"],
    ["Belum Ujian (APPROVE INI)", len(not_found_df), f"{len(not_found_df)/len(not_approved_df)*100:.1f}%" if len(not_approved_df) > 0 else "0%", "✅"],
]
for row in summary:
    ws1.append(row)
style_worksheet(ws1, 4, "✅", "❌")

# Sheet 2: APPROVE
ws2 = wb.create_sheet("APPROVE - Belum Ujian")
if len(not_found_df) > 0:
    for r in dataframe_to_rows(not_found_df[['Nama Peserta', 'NIM', 'Jurusan', 'Sumber Data', 'Program', 'Rekomendasi']], index=False, header=True):
        ws2.append(r)
    style_worksheet(ws2)
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.fill = approve_fill

# Sheet 3: JANGAN APPROVE
ws3 = wb.create_sheet("JANGAN APPROVE - Sudah Ujian")
if len(found_df) > 0:
    for r in dataframe_to_rows(found_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Nama di Certiport', 'Skor Kecocokan', 'Rekomendasi']], index=False, header=True):
        ws3.append(r)
    style_worksheet(ws3)
    for row in ws3.iter_rows(min_row=2, max_row=ws3.max_row):
        for cell in row:
            cell.fill = reject_fill

# Sheet 4: Approved Filter (VERIFIKASI PESERTA APPROVED)
ws4 = wb.create_sheet("Approved Filter")
if len(approved_filter_df) > 0:
    cols_to_show = ['Nama Peserta', 'NIM', 'Jurusan', 'Sumber Data', 'Program', 
                   'Nama di Certiport', 'Skor Kecocokan', 'Status Verifikasi']
    for r in dataframe_to_rows(approved_filter_df[cols_to_show], index=False, header=True):
        ws4.append(r)
    style_worksheet(ws4)
    # Apply coloring: Red for those found in Certiport (should not be approved), Green for OK
    for row_idx, row in enumerate(ws4.iter_rows(min_row=2, max_row=ws4.max_row), start=2):
        status_val = str(ws4.cell(row=row_idx, column=8).value)  # Status Verifikasi column
        if 'PERLU DICEK' in status_val or 'ADA DI CERTIPORT' in status_val:
            for cell in row:
                cell.fill = reject_fill  # Red fill for warning
        else:
            for cell in row:
                cell.fill = approve_fill  # Green fill for OK

# Sheet 5: Semua Hasil Cross-Check
ws5 = wb.create_sheet("Semua Hasil Cross-Check")
for r in dataframe_to_rows(results_df, index=False, header=True):
    ws5.append(r)
style_worksheet(ws5, 10, "TIDAK DITEMUKAN", "DITEMUKAN")

# Simpan file
excel_file = 'HASIL_FILTER_PESERTA_LENGKAP.xlsx'
wb.save(excel_file)

print(f"\n✅ File Excel berhasil disimpan: {excel_file}")
print(f"\n📋 DAFTAR SHEETS:")
print(f"   1. Dashboard Summary          - Ringkasan keseluruhan")
print(f"   2. APPROVE - Belum Ujian      - 🟢 {len(not_found_df)} peserta")
print(f"   3. JANGAN APPROVE - Sudah     - 🔴 {len(found_df)} peserta")
print(f"   4. Approved Filter            - 🔍 Verifikasi APPROVED ({len(approved_warning_df)} perlu dicek)")
print(f"   5. Semua Hasil Cross-Check    - Detail lengkap")

📊 EXPORT KE EXCEL DENGAN MULTIPLE SHEETS

✅ File Excel berhasil disimpan: HASIL_FILTER_PESERTA_LENGKAP.xlsx

📋 DAFTAR SHEETS:
   1. Dashboard Summary          - Ringkasan keseluruhan
   2. APPROVE - Belum Ujian      - 🟢 10 peserta
   3. JANGAN APPROVE - Sudah     - 🔴 62 peserta
   4. Approved Filter            - 🔍 Verifikasi APPROVED (1 perlu dicek)
   5. Semua Hasil Cross-Check    - Detail lengkap


## 🔍 Fungsi Pencarian Manual

In [39]:
# ============================================================
# FUNGSI PENCARIAN MANUAL
# ============================================================

def cari_nama(nama, sumber='MOS'):
    """
    Fungsi untuk mencari nama secara manual di database Certiport
    
    Parameter:
    - nama: nama yang ingin dicari
    - sumber: 'MCF' atau 'MOS'
    
    Contoh: cari_nama('JOHN DOE', 'MOS')
    """
    if sumber == 'MCF':
        names_to_search = certiport_mcf_names
    else:
        names_to_search = certiport_mos_names
    
    match, score, match_type = find_best_match(nama, names_to_search)
    
    print(f"\n🔍 Hasil Pencarian untuk: {nama}")
    print(f"   Sumber: {sumber}")
    print(f"   {'='*50}")
    if match:
        print(f"   ✅ DITEMUKAN!")
        print(f"   Nama di Certiport: {match}")
        print(f"   Skor Kecocokan: {score}")
        print(f"   Tipe Match: {match_type}")
    else:
        print(f"   ❌ TIDAK DITEMUKAN")
        print(f"   Skor Tertinggi: {score}")

print("✅ Fungsi cari_nama() siap digunakan!")
print("\n📖 Cara Pakai:")
print("   cari_nama('NAMA LENGKAP', 'MOS')  # Untuk MOS")
print("   cari_nama('NAMA LENGKAP', 'MCF')  # Untuk MCF")

✅ Fungsi cari_nama() siap digunakan!

📖 Cara Pakai:
   cari_nama('NAMA LENGKAP', 'MOS')  # Untuk MOS
   cari_nama('NAMA LENGKAP', 'MCF')  # Untuk MCF


In [40]:
# ============================================================
# CONTOH PENGGUNAAN FUNGSI PENCARIAN
# ============================================================

# Contoh: cari_nama('MUHAMMAD REVIANSYAH', 'MOS')